# 10 — StationXML + AK events (cache-aware)

Run this after `00_config.ipynb`.

In [ ]:
from obspy import read_inventory

%run 00_config.ipynb

In [ ]:
# --- StationXML (subset around Redoubt) ---
stationxml_path = os.path.join(ROOT, "Redoubt_0p2deg_StationXML.xml")

if file_exists(stationxml_path):
    print("StationXML exists, reading:", stationxml_path)
    inv = read_inventory(stationxml_path)
else:
    print("Downloading StationXML subset...")
    inv = EARTHSCOPE.get_stations(
        latitude=REDOUBT_LAT, longitude=REDOUBT_LON,
        maxradius=RADIUS_DEG,
        starttime=t0, endtime=t1,
        level="response",
    )
    inv.write(stationxml_path, format="STATIONXML")
    print("Wrote:", stationxml_path)

print(inv)


In [ ]:
# --- Events from USGS (AK catalog) ---
from obspy import read_events

quakeml_path = os.path.join(ROOT, "AK_events_Redoubt_0p2deg_20090320_23.xml")

if file_exists(quakeml_path):
    print("Events file exists, reading:", quakeml_path)
    cat = read_events(quakeml_path)
else:
    print("Downloading events from USGS AK catalog...")
    cat = USGS.get_events(
        starttime=t0, endtime=t1,
        latitude=REDOUBT_LAT, longitude=REDOUBT_LON,
        maxradius=RADIUS_DEG,
        catalog="ak",
        includeallorigins=True,
        includeallmagnitudes=True,
        limit=20000
    )
    cat.write(quakeml_path, format="QUAKEML")
    print("Wrote:", quakeml_path)

print("N events:", len(cat))


In [ ]:
# --- Convert to DataFrame for plotting ---
rows = []
for ev in cat:
    org = ev.preferred_origin() or (ev.origins[0] if ev.origins else None)
    mag = ev.preferred_magnitude() or (ev.magnitudes[0] if ev.magnitudes else None)
    if org is None:
        continue
    rows.append(dict(
        time=org.time.datetime,
        lat=org.latitude,
        lon=org.longitude,
        depth_km=(org.depth or np.nan)/1000.0,
        mag=(mag.mag if mag else np.nan),
    ))

df_events = pd.DataFrame(rows).sort_values("time")
display(df_events.head())
print("Events with origins:", len(df_events))


In [ ]:
# --- Plot: event locations ---
plt.figure()
plt.scatter(df_events["lon"], df_events["lat"], s=10)
plt.scatter([REDOUBT_LON], [REDOUBT_LAT], marker="*", s=150)
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.title("AK events near Redoubt (0.2°)")
plt.show()


In [ ]:
# --- Plot: magnitude vs time ---
plt.figure()
plt.scatter(df_events["time"], df_events["mag"], s=10)
plt.xlabel("Time"); plt.ylabel("Magnitude")
plt.title("Magnitude vs time (AK catalog subset)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
